# 06 · Ask it twice: the same numbers in SQL and in MQL
Each question is answered **twice, by two different tools from two different tables**:

| | SQL | MQL |
|---|---|---|
| Engine | DuckDB | MongoDB Atlas (aggregation pipeline) |
| Reads | the 3.85M-row `cascades` gold table (Parquet) | the small collections loaded by notebook 05 |

In [2]:
import os, json
from pathlib import Path
import pandas as pd, duckdb
from pymongo import MongoClient

PROJECT = Path(os.path.expanduser("~/Desktop/ADSP32030_Scalable/Project"))
CASCADES = PROJECT/"data/propagation/cascades"
QUERIES  = PROJECT/"queries"; QUERIES.mkdir(exist_ok=True)

env = {}
if (PROJECT/".env").exists():
    for line in (PROJECT/".env").read_text().splitlines():
        if line.strip() and not line.lstrip().startswith("#") and "=" in line:
            k, v = line.split("=", 1); env[k.strip()] = v.strip()
URI, DBNAME = env.get("ATLAS_URI"), env.get("ATLAS_DB", "flightdelay")
DRY_RUN = not URI
db = None if DRY_RUN else MongoClient(URI, serverSelectionTimeoutMS=15000)[DBNAME]
con = duckdb.connect()
sql = lambda text: con.execute(text.replace("{CASCADES}", f"read_parquet('{CASCADES}/*.parquet')")).df()
print("connected to Atlas" if db is not None else "DRY RUN: SQL side only")

connected to Atlas


## The five question pairs
Each pair is `(name, SQL, MQL pipeline, collection)`. The SQL runs on the full `cascades` table, the pipeline on the loaded summary collections.

In [3]:
PAIRS = {
"q1_top_airports": dict(
    question="Which 10 airports start the most cascades?",
    sql="""SELECT root_origin AS airport, count(*) AS cascades
FROM {CASCADES} GROUP BY 1 ORDER BY cascades DESC, airport LIMIT 10""",
    coll="summary_airport_hour",
    mql=[{"$group": {"_id": "$airport", "cascades": {"$sum": "$cascades"}}},
         {"$sort": {"cascades": -1, "_id": 1}}, {"$limit": 10},
         {"$project": {"_id": 0, "airport": "$_id", "cascades": 1}}], key="airport"),
"q2_by_hour": dict(
    question="How many cascades start in each local hour?",
    sql="""SELECT root_local_hour AS local_hour, count(*) AS cascades
FROM {CASCADES} GROUP BY 1 ORDER BY 1""",
    coll="summary_airport_hour",
    mql=[{"$group": {"_id": "$local_hour", "cascades": {"$sum": "$cascades"}}}, {"$sort": {"_id": 1}},
         {"$project": {"_id": 0, "local_hour": "$_id", "cascades": 1}}], key="local_hour"),
"q3_by_year": dict(
    question="How many cascades per year?",
    sql="""SELECT Year, count(*) AS cascades FROM {CASCADES} GROUP BY 1 ORDER BY 1""",
    coll="by_year",
    mql=[{"$project": {"_id": 0, "Year": 1, "cascades": 1}}, {"$sort": {"Year": 1}}], key="Year"),
"q4_total": dict(
    question="How many cascades in total, and what is their average length?",
    sql="""SELECT count(*) AS cascades, avg(chain_len) AS avg_chain_len FROM {CASCADES}""",
    coll="summary_airport_hour",
    mql=[{"$group": {"_id": None, "cascades": {"$sum": "$cascades"},
                     "weighted_len": {"$sum": {"$multiply": ["$avg_chain_len", "$cascades"]}}}},
         {"$project": {"_id": 0, "cascades": 1, "avg_chain_len": {"$divide": ["$weighted_len", "$cascades"]}}}], key=None),
"q5_dfw_busy_hours": dict(
    question="At DFW, which 3 local hours start the most cascades?",
    sql="""SELECT root_local_hour AS local_hour, count(*) AS cascades
FROM {CASCADES} WHERE root_origin = 'DFW' GROUP BY 1 ORDER BY cascades DESC, local_hour LIMIT 3""",
    coll="summary_airport_hour",
    mql=[{"$match": {"airport": "DFW"}}, {"$sort": {"cascades": -1, "local_hour": 1}}, {"$limit": 3},
         {"$project": {"_id": 0, "local_hour": 1, "cascades": 1}}], key="local_hour"),
}

# save every query as a file for the repo
for name, p in PAIRS.items():
    (QUERIES/f"{name}.sql").write_text(f"-- {p['question']}\n-- DuckDB over the gold table data/propagation/cascades (Parquet)\n{p['sql']};\n")
    (QUERIES/f"{name}.mql.json").write_text(json.dumps({"question": p["question"], "collection": p["coll"], "pipeline": p["mql"]}, indent=2) + "\n")
print("saved", len(PAIRS), "SQL files and", len(PAIRS), "MQL files to", QUERIES)

saved 5 SQL files and 5 MQL files to /Users/a../Desktop/ADSP32030_Scalable/Project/queries


## Run both and compare

In [4]:
results, all_ok = [], True
for name, p in PAIRS.items():
    a = sql(p["sql"])
    print(f"\n== {name}: {p['question']}\n-- SQL (DuckDB, Parquet)"); print(a.to_string(index=False))
    if DRY_RUN: continue
    b = pd.DataFrame(list(db[p["coll"]].aggregate(p["mql"])))
    print("-- MQL (Atlas)"); print(b.to_string(index=False))
    if p["key"]:
        b = b[a.columns.tolist()] if set(a.columns) <= set(b.columns) else b
        same = a.shape == b.shape and (a.reset_index(drop=True).astype(float).values == b.reset_index(drop=True).astype(float).values).all() \
               if p["key"] != "airport" else a.equals(b.reset_index(drop=True))
    else:
        same = int(a.cascades[0]) == int(b.cascades[0]) and abs(float(a.avg_chain_len[0]) - float(b.avg_chain_len[0])) < 0.01
    all_ok &= bool(same); results.append((name, bool(same)))
    print("AGREE" if same else "DIFFERENT")
if not DRY_RUN:
    print("\n" + "\n".join(f"{'OK  ' if ok else 'FAIL'} {n}" for n, ok in results))
    print("\nALL FIVE QUESTIONS GIVE THE SAME ANSWER IN SQL AND MQL" if all_ok else "\nSOME ANSWERS DIFFER: investigate before using them")


== q1_top_airports: Which 10 airports start the most cascades?
-- SQL (DuckDB, Parquet)
airport  cascades
    DFW    272540
    ATL    262857
    ORD    260877
    DEN    193265
    CLT    129650
    PHX    107154
    IAH    105299
    LAX    104016
    LAS     97905
    MCO     97315
-- MQL (Atlas)
 cascades airport
   272540     DFW
   262857     ATL
   260877     ORD
   193265     DEN
   129650     CLT
   107154     PHX
   105299     IAH
   104016     LAX
    97905     LAS
    97315     MCO
AGREE

== q2_by_hour: How many cascades start in each local hour?
-- SQL (DuckDB, Parquet)
 local_hour  cascades
          0      1996
          1       682
          2       186
          3       207
          4       154
          5     44811
          6    188170
          7    262208
          8    328711
          9    287016
         10    302621
         11    289132
         12    287865
         13    317788
         14    330329
         15    345664
         16    307715
         17  